### np.where

In [1]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array detected: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def flatten(a):
    if not isinstance(a, list):
        return [a]
    result = []
    for elem in a:
        result.extend(flatten(elem))
    return result


def _product(shape):
    p = 1
    for s in shape:
        p *= s
    return p


def reshape(flat, shape):
    if len(shape) == 0:
        if len(flat) != 1:
            raise ValueError("Cannot reshape to scalar")
        return flat[0]

    if len(shape) == 1:
        if len(flat) != shape[0]:
            raise ValueError("Cannot reshape: size mismatch")
        return flat[:]

    chunk_size = _product(shape[1:])
    if len(flat) != shape[0] * chunk_size:
        raise ValueError("Cannot reshape: size mismatch")

    result = []
    for i in range(shape[0]):
        start = i * chunk_size
        end = (i + 1) * chunk_size
        result.append(reshape(flat[start:end], shape[1:]))
    return result


def get_ndim(a):
    return len(get_shape(a))


def _normalize_axis(axis, ndim):
    if axis < 0:
        axis += ndim
    if axis < 0 or axis >= ndim:
        raise ValueError(f"axis {axis} is out of bounds for array of dimension {ndim}")
    return axis


def _coords_from_flat_index(idx, shape):
    if len(shape) == 0:
        return ()
    coords = []
    for dim in reversed(shape):
        coords.append(idx % dim)
        idx //= dim
    coords.reverse()
    return tuple(coords)


def _broadcast_shape(shape1, shape2):
    result = []
    i = len(shape1) - 1
    j = len(shape2) - 1

    while i >= 0 or j >= 0:
        d1 = shape1[i] if i >= 0 else 1
        d2 = shape2[j] if j >= 0 else 1

        if d1 == d2 or d1 == 1 or d2 == 1:
            result.append(max(d1, d2))
        else:
            raise ValueError(f"Shapes {shape1} and {shape2} are not broadcastable")

        i -= 1
        j -= 1

    result.reverse()
    return tuple(result)


def _broadcast_three_shapes(shape1, shape2, shape3):
    return _broadcast_shape(_broadcast_shape(shape1, shape2), shape3)


def _broadcast_index(out_idx, out_shape, in_shape):
    if len(in_shape) == 0:
        return ()

    offset = len(out_shape) - len(in_shape)
    idx = []

    for i in range(len(in_shape)):
        in_dim = in_shape[i]
        out_dim_index = i + offset
        out_coord = out_idx[out_dim_index]

        if in_dim == 1:
            idx.append(0)
        else:
            idx.append(out_coord)

    return tuple(idx)


def _get_item(a, idx):
    cur = a
    for i in idx:
        cur = cur[i]
    return cur


def _build_nested(shape, getter, prefix=()):
    if len(shape) == 0:
        return getter(prefix)
    return [_build_nested(shape[1:], getter, prefix + (i,)) for i in range(shape[0])]


def _where_condition_only_1d(condition):
    result = []
    for i, value in enumerate(condition):
        if value:
            result.append(i)
    return (result,)


def _where_condition_only_nd(condition, shape):
    flat = flatten(condition)
    ndim = len(shape)
    indices = [[] for _ in range(ndim)]

    for flat_idx, value in enumerate(flat):
        if value:
            coords = _coords_from_flat_index(flat_idx, shape)
            for axis in range(ndim):
                indices[axis].append(coords[axis])

    return tuple(indices)


def py_where(condition, x=None, y=None):
    cond_shape = get_shape(condition)
    cond_ndim = len(cond_shape)

    if x is None and y is None:
        if cond_ndim == 0:
            return ([],) if not condition else ([0],)
        if cond_ndim == 1:
            return _where_condition_only_1d(condition)
        return _where_condition_only_nd(condition, cond_shape)

    if (x is None) != (y is None):
        raise ValueError("Either both or neither of x and y should be given")

    x_shape = get_shape(x)
    y_shape = get_shape(y)

    out_shape = _broadcast_three_shapes(cond_shape, x_shape, y_shape)

    def getter(out_idx):
        cond_idx = _broadcast_index(out_idx, out_shape, cond_shape)
        x_idx = _broadcast_index(out_idx, out_shape, x_shape)
        y_idx = _broadcast_index(out_idx, out_shape, y_shape)

        cond_val = _get_item(condition, cond_idx) if len(cond_shape) > 0 else condition
        x_val = _get_item(x, x_idx) if len(x_shape) > 0 else x
        y_val = _get_item(y, y_idx) if len(y_shape) > 0 else y

        return x_val if cond_val else y_val

    return _build_nested(out_shape, getter)

### tets cases 

In [2]:
print(py_where([True, False, True], [1, 2, 3], [10, 20, 30]))

print(py_where([[True, False], [False, True]], [[1, 2], [3, 4]], [[9, 8], [7, 6]]))

print(py_where([True, False, True]))

print(py_where([[True, False], [False, True]]))

print(py_where([True, False, True], 100, 0))

print(py_where([[True], [False]], [[1, 2, 3], [4, 5, 6]], 0))

print(py_where(True, 5, 9))

print(py_where(False, 5, 9))

print(py_where([], [], []))

print(py_where([False, False, False]))

[1, 20, 3]
[[1, 8], [7, 4]]
([0, 2],)
([0, 1], [0, 1])
[100, 0, 100]
[[1, 2, 3], [0, 0, 0]]
5
9
[]
([],)


In [3]:
print(py_where([True, False], [1, 2], None))

ValueError: Either both or neither of x and y should be given

In [4]:
print(py_where([[True, False], [True]], [[1, 2], [3, 4]], [[5, 6], [7, 8]]))

ValueError: Jagged array detected: first row shape (2,), but row 1 shape (1,)

In [5]:
print(py_where([True, False], [1, 2, 3], [4, 5]))

ValueError: Shapes (2,) and (3,) are not broadcastable